# Ruido blanco — señal y espectro
> FRA — visualización de la señal de excitación para el modo Bode

El ruido blanco tiene **espectro plano**: igual energía en todas las frecuencias.
Por eso es ideal para medir la respuesta en frecuencia de un sistema — se excita todo el espectro de interés (20Hz–20kHz) de una sola vez.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import welch

plt.style.use('dark_background')
ACCENT  = '#4a9eff'
ACCENT2 = '#4ecb71'
GRAY    = '#445566'

In [ ]:
# ── Parámetros ──────────────────────────────────────────────────
FS      = 96_000   # Hz — sample rate del ADC del FRA
DURACION = 0.05    # segundos — suficiente para ver la señal y el espectro
F_MIN   = 20       # Hz — límite inferior del espectro de interés
F_MAX   = 20_000   # Hz — límite superior

N = int(FS * DURACION)
t = np.arange(N) / FS  # eje de tiempo en segundos

# Ruido blanco gaussiano, normalizado a amplitud ±1
np.random.seed(42)
ruido = np.random.randn(N)
ruido /= np.max(np.abs(ruido))

print(f'Muestras: {N}  |  Duración: {DURACION*1000:.0f} ms  |  Fs: {FS/1000:.0f} kHz')

## Señal en el tiempo

In [ ]:
# Mostramos solo los primeros 5ms para que sea legible
VENTANA_MS = 5
n_ventana  = int(FS * VENTANA_MS / 1000)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t[:n_ventana] * 1000, ruido[:n_ventana], color=ACCENT, lw=0.7, alpha=0.9)
ax.axhline(0, color=GRAY, lw=0.5, ls='--')
ax.set_xlabel('Tiempo (ms)', color='#aaa')
ax.set_ylabel('Amplitud (normalizada)', color='#aaa')
ax.set_title('Ruido blanco — dominio del tiempo (primeros 5 ms)', color='#ccc', pad=12)
ax.set_xlim(0, VENTANA_MS)
ax.set_ylim(-1.2, 1.2)
ax.tick_params(colors='#667')
for spine in ax.spines.values():
    spine.set_edgecolor('#223')
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#0d0d1a')
plt.tight_layout()
plt.show()

## Espectro de frecuencias (densidad espectral de potencia)

Usamos el método de Welch (promedio de ventanas solapadas) para estimar el espectro — más suave y estable que la FFT directa de una sola ventana.

In [ ]:
f, Pxx = welch(ruido, fs=FS, nperseg=2048, noverlap=1024, window='hann')
Pxx_db = 10 * np.log10(Pxx + 1e-12)  # a dB

# Máscara: solo mostrar el rango de interés
mask = (f >= F_MIN) & (f <= F_MAX)

fig, ax = plt.subplots(figsize=(12, 4))
ax.semilogx(f[mask], Pxx_db[mask], color=ACCENT2, lw=1.2)
ax.fill_between(f[mask], Pxx_db[mask], Pxx_db[mask].min() - 5,
                color=ACCENT2, alpha=0.08)

# Línea de referencia — nivel medio
nivel_medio = np.mean(Pxx_db[mask])
ax.axhline(nivel_medio, color='#f0a020', lw=1, ls='--', alpha=0.6, label=f'Nivel medio: {nivel_medio:.1f} dB')

# Marcadores de frecuencias clave
for fc, label in [(20, '20 Hz'), (1000, '1 kHz'), (10_000, '10 kHz'), (20_000, '20 kHz')]:
    ax.axvline(fc, color=GRAY, lw=0.6, ls=':')
    ax.text(fc, ax.get_ylim()[0] if ax.get_ylim()[0] > -200 else nivel_medio - 20,
            label, color='#556', fontsize=8, ha='center', va='bottom')

ax.set_xscale('log')
ax.set_xlim(F_MIN, F_MAX)
ax.set_xlabel('Frecuencia (Hz)', color='#aaa')
ax.set_ylabel('PSD (dB/Hz)', color='#aaa')
ax.set_title('Ruido blanco — espectro de frecuencias (20 Hz – 20 kHz)', color='#ccc', pad=12)
ax.legend(facecolor='#111', edgecolor='#333', labelcolor='#aaa', fontsize=9)
ax.tick_params(colors='#667')
ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'], color='#667')
for spine in ax.spines.values():
    spine.set_edgecolor('#223')
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#0d0d1a')
plt.tight_layout()
plt.show()

## Vista combinada — señal + espectro

In [ ]:
fig = plt.figure(figsize=(14, 6), facecolor='#0d0d1a')
gs  = gridspec.GridSpec(2, 1, hspace=0.45)

# — Señal temporal —
ax1 = fig.add_subplot(gs[0])
ax1.plot(t[:n_ventana] * 1000, ruido[:n_ventana], color=ACCENT, lw=0.7)
ax1.axhline(0, color=GRAY, lw=0.5, ls='--')
ax1.set_xlim(0, VENTANA_MS)
ax1.set_ylim(-1.3, 1.3)
ax1.set_xlabel('Tiempo (ms)', color='#888', fontsize=9)
ax1.set_ylabel('Amplitud', color='#888', fontsize=9)
ax1.set_title('Señal en el tiempo', color='#aaa', fontsize=10, pad=8)
ax1.tick_params(colors='#556', labelsize=8)
ax1.set_facecolor('#0d0d1a')
for spine in ax1.spines.values(): spine.set_edgecolor('#1a2030')

# Anotación
ax1.annotate('señal no periódica,\ncambia cada instante',
             xy=(2.5, ruido[int(2.5e-3*FS)]), xytext=(3.2, 0.7),
             color='#556', fontsize=8,
             arrowprops=dict(arrowstyle='->', color='#334', lw=0.8))

# — Espectro —
ax2 = fig.add_subplot(gs[1])
ax2.semilogx(f[mask], Pxx_db[mask], color=ACCENT2, lw=1.2)
ax2.fill_between(f[mask], Pxx_db[mask], Pxx_db[mask].min() - 5,
                 color=ACCENT2, alpha=0.08)
ax2.axhline(nivel_medio, color='#f0a020', lw=1, ls='--', alpha=0.7,
            label='nivel medio (≈ plano)')

# Banda sombreada 20Hz-20kHz
ax2.axvspan(F_MIN, F_MAX, color='#4a9eff', alpha=0.03)

ax2.set_xlim(F_MIN, F_MAX)
ax2.set_xlabel('Frecuencia (Hz)', color='#888', fontsize=9)
ax2.set_ylabel('PSD (dB/Hz)', color='#888', fontsize=9)
ax2.set_title('Espectro de frecuencias — banda audible', color='#aaa', fontsize=10, pad=8)
ax2.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
ax2.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'],
                    color='#667', fontsize=8)
ax2.tick_params(colors='#556', labelsize=8)
ax2.legend(facecolor='#0d0d1a', edgecolor='#1a2030', labelcolor='#aaa', fontsize=8)
ax2.set_facecolor('#0d0d1a')
for spine in ax2.spines.values(): spine.set_edgecolor('#1a2030')

# Anotación espectro plano
ax2.annotate('espectro plano:\nigual energía en toda la banda',
             xy=(5000, nivel_medio + 1), xytext=(300, nivel_medio + 8),
             color='#556', fontsize=8,
             arrowprops=dict(arrowstyle='->', color='#334', lw=0.8))

plt.suptitle('Ruido blanco — excitación para medición de Bode', 
             color='#7eb8f7', fontsize=12, y=1.01)
plt.savefig('ruido_blanco.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('Figura guardada: ruido_blanco.png')

## ¿Por qué ruido blanco para medir Bode?

| Señal de excitación | Cobertura espectral | Velocidad | Uso típico |
|---|---|---|---|
| **Ruido blanco** | Toda la banda de una vez | Rápida | FRA, medición rápida |
| Swept sine (barrido) | Una frecuencia a la vez | Lenta | Alta precisión |
| Chirp | Toda la banda, ordenada | Media | Audímetros, sonar |

El FRA usa ruido blanco porque:
- Excita **todas las frecuencias simultáneamente** → un solo bloque de señal alcanza para calcular todo el Bode
- Es simple de generar digitalmente (solo `randn()`)
- El espectro plano garantiza que no hay frecuencias subexcitadas

## ¿Qué tan plano es? — Análisis de planitud

Tres métricas para cuantificarlo:

| Métrica | Qué mide | Referencia para el FRA |
|---|---|---|
| **SFM** (Spectral Flatness Measure) | Cuánto se parece al ruido blanco ideal | > −1 dB = suficiente |
| **Desv. estándar (dB)** | Variabilidad bin a bin del espectro | < 3 dB = aceptable para audio |
| **Desviación máxima** | El peor pico/valle respecto a la media | < 6 dB = tolerable |

La clave: más duración de la señal → más ventanas de promedio en Welch → espectro más suave → más plano.

In [ ]:
def sfm_db(Pxx):
    """Spectral Flatness Measure en dB. 0 dB = perfectamente plano."""
    log_mean = np.mean(np.log(Pxx + 1e-30))
    lin_mean = np.mean(Pxx)
    return 10 * np.log10(np.exp(log_mean) / (lin_mean + 1e-30))

def analizar_planitud(duracion_s, label):
    n     = int(FS * duracion_s)
    sig   = np.random.randn(n)
    sig  /= np.max(np.abs(sig))
    ff, P = welch(sig, fs=FS, nperseg=2048, noverlap=1024, window='hann')
    m     = (ff >= F_MIN) & (ff <= F_MAX)
    P_db  = 10 * np.log10(P[m] + 1e-12)
    n_ventanas = (n - 1024) // 1024   # aprox ventanas de promedio

    sfm        = sfm_db(P[m])
    std_db     = np.std(P_db)
    max_dev_db = np.max(np.abs(P_db - np.mean(P_db)))

    # veredictos por métrica
    ok_sfm = sfm > -1.0
    ok_std = std_db < 3.0
    ok_dev = max_dev_db < 6.0
    apto   = ok_sfm and ok_std and ok_dev

    print(f"\n{'─'*52}")
    print(f"  {label}  ({duracion_s*1000:.0f} ms · ~{n_ventanas} ventanas Welch)")
    print(f"{'─'*52}")
    print(f"  SFM            : {sfm:+.2f} dB   {'✓' if ok_sfm else '✗'}  (ref > -1 dB)")
    print(f"  Std espectro   : {std_db:.2f} dB    {'✓' if ok_std else '✗'}  (ref < 3 dB)")
    print(f"  Desv. máxima   : {max_dev_db:.2f} dB    {'✓' if ok_dev else '✗'}  (ref < 6 dB)")
    print(f"  ➜  {'APTO para el FRA ✓' if apto else 'INSUFICIENTE — aumentar duración ✗'}")
    return ff[m], P_db

# Comparar distintas duraciones de señal de excitación
np.random.seed(42)
resultados = {}
for dur, lbl in [(0.05, '50 ms'), (0.1, '100 ms'), (0.5, '500 ms'), (1.0, '1 s')]:
    resultados[lbl] = analizar_planitud(dur, lbl)

In [ ]:
# Visualización: espectros de cada duración con bandas ±3 dB
colores = ['#ff6060', '#f0a020', '#4a9eff', '#4ecb71']
labels  = list(resultados.keys())

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#0d0d1a')
fig.suptitle('Planitud del espectro según duración de la señal', color='#7eb8f7', fontsize=13)

for ax, (lbl, (ff, P_db)), color in zip(axes.flat, resultados.items(), colores):
    media = np.mean(P_db)
    ax.semilogx(ff, P_db, color=color, lw=0.9, alpha=0.85, label='PSD')
    ax.axhline(media,       color='white',   lw=1.0, ls='--', alpha=0.5, label='media')
    ax.axhline(media + 3,   color='#f0a020', lw=0.8, ls=':',  alpha=0.7, label='±3 dB')
    ax.axhline(media - 3,   color='#f0a020', lw=0.8, ls=':',  alpha=0.7)
    ax.fill_between(ff, media - 3, media + 3, color='#f0a020', alpha=0.05)

    std_v   = np.std(P_db)
    max_dev = np.max(np.abs(P_db - media))
    apto    = std_v < 3.0 and max_dev < 6.0

    veredicto = '✓ APTO' if apto else '✗ INSUFICIENTE'
    color_v   = '#4ecb71' if apto else '#ff6060'

    ax.set_title(f'{lbl}   —   std={std_v:.1f} dB  |  max dev={max_dev:.1f} dB',
                 color='#aaa', fontsize=9)
    ax.text(0.98, 0.05, veredicto, transform=ax.transAxes,
            ha='right', va='bottom', color=color_v, fontsize=11, fontweight='bold')

    ax.set_xlim(F_MIN, F_MAX)
    ax.set_ylim(media - 15, media + 15)
    ax.set_xticks([20, 200, 2000, 20000])
    ax.set_xticklabels(['20Hz', '200Hz', '2kHz', '20kHz'], color='#667', fontsize=8)
    ax.tick_params(colors='#556', labelsize=8)
    ax.set_facecolor('#0d0d1a')
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2030')
    if ax == axes.flat[0]:
        ax.legend(facecolor='#0d0d1a', edgecolor='#1a2030', labelcolor='#aaa', fontsize=8)

plt.tight_layout()
plt.show()

## Conclusión para el FRA

El análisis anterior muestra cuánta señal necesita el FRA para una medición válida:

- **50 ms** → espectro rugoso, probablemente insuficiente
- **100 ms** → borderline, depende de la realización
- **500 ms** → std < 1 dB, apto con margen
- **1 s** → casi perfectamente plano, referencia de calidad

**Recomendación para el firmware del FRA:** usar bloques de **500 ms** como duración mínima de la señal de ruido blanco en modo Bode. Esto da ~46 ventanas de promedio en Welch, suficiente para std < 1.5 dB en toda la banda.

> **Nota importante:** aunque el espectro no sea perfecto, la medición de Bode sigue siendo válida porque `H(f) = ADC₂(f) / ADC₁(f)` — cualquier irregularidad del ruido en ADC₁ se divide y cancela en ADC₂. Lo que importa es que el SNR sea suficiente en todo el rango, no que el espectro sea perfecto.

---
## Cómo se extrae la fase — Demo completa de medición de Bode

La FFT de cualquier señal devuelve **números complejos**. Cada bin tiene:
- **Magnitud** → `|FFT(f)|` — cuánta energía hay en esa frecuencia
- **Fase** → `angle(FFT(f))` — en qué punto del ciclo está esa frecuencia

La función de transferencia compleja es:

```
H(f) = FFT(salida) / FFT(entrada)   →   número complejo

|H(f)| en dB  = 20·log10(|H(f)|)        ← Bode magnitud
∠H(f) en °    = angle(H(f)) × 180/π     ← Bode fase
```

**Estimador H1 (más robusto con ruido):**
En lugar de dividir FFTs directamente (sensible al ruido), se usa el espectro cruzado:

```
H(f) = Gxy(f) / Gxx(f)

Gxy = FFT*(entrada) × FFT(salida)   ← espectro cruzado (complejo)
Gxx = |FFT(entrada)|²               ← autoespectro de entrada (real)
```

Promediando Gxy y Gxx en múltiples ventanas (Welch), el ruido se promedia y la señal se refuerza.

In [ ]:
from scipy.signal import csd, coherence, lfilter, freqs

# ── Sistema bajo prueba: filtro RC pasa-bajos con fc = 1 kHz ────
FC = 1000  # Hz — frecuencia de corte
RC = 1 / (2 * np.pi * FC)

# Respuesta teórica: H(f) = 1 / (1 + j·2π·f·RC)
f_teo  = np.logspace(np.log10(20), np.log10(20000), 500)
H_teo  = 1 / (1 + 1j * 2 * np.pi * f_teo * RC)
mag_teo   = 20 * np.log10(np.abs(H_teo))
phase_teo = np.angle(H_teo) * 180 / np.pi

# ── Señal de prueba: ruido blanco (500 ms) ──────────────────────
DUR_BODE = 0.5
N_b = int(FS * DUR_BODE)
np.random.seed(7)
x = np.random.randn(N_b)   # entrada: ruido blanco

# Simular el filtro RC discreto: H(z) = (1-a) / (1 - a·z⁻¹),  a = exp(-1/(RC·Fs))
a  = np.exp(-1 / (RC * FS))
b_filt, a_filt = [1 - a], [1, -a]
y = lfilter(b_filt, a_filt, x)  # salida: x filtrado

# Agregar ruido de medición realista en la salida (SNR ~40 dB)
snr_lineal = 10 ** (40 / 20)
y += np.random.randn(N_b) * np.std(y) / snr_lineal

print(f"Señal de prueba: {DUR_BODE*1000:.0f} ms  |  {N_b} muestras  |  fc teórica: {FC} Hz")
print(f"Ruido de medición añadido: SNR = 40 dB")

In [ ]:
NPERSEG = 2048

# ── Estimador H1: Gxy / Gxx ─────────────────────────────────────
f_m, Gxy = csd(x, y, fs=FS, nperseg=NPERSEG, noverlap=NPERSEG//2, window='hann')
f_m, Gxx = welch(x,    fs=FS, nperseg=NPERSEG, noverlap=NPERSEG//2, window='hann')
f_m, coh = coherence(x, y, fs=FS, nperseg=NPERSEG, noverlap=NPERSEG//2, window='hann')

H_med = Gxy / Gxx          # función de transferencia compleja medida

mag_med   = 20 * np.log10(np.abs(H_med) + 1e-12)
phase_med = np.unwrap(np.angle(H_med)) * 180 / np.pi   # unwrap evita saltos de ±180°

# Solo banda de interés
mask = (f_m >= F_MIN) & (f_m <= F_MAX)

# ── Gráfica Bode completa ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.patch.set_facecolor('#0d0d1a')
fig.suptitle(f'Bode medido vs teórico — Filtro RC pasa-bajos  fc = {FC} Hz',
             color='#7eb8f7', fontsize=13)

# — Magnitud —
ax = axes[0]
ax.semilogx(f_teo,    mag_teo,           color='#f0a020', lw=2,   ls='--', label='Teórico')
ax.semilogx(f_m[mask], mag_med[mask],    color=ACCENT2,   lw=1.2,          label='Medido (FRA)')
ax.axvline(FC, color='#ff6060', lw=0.8, ls=':', alpha=0.7)
ax.text(FC*1.05, 2, f'fc = {FC} Hz', color='#ff6060', fontsize=8)
ax.axhline(-3, color='#556', lw=0.6, ls=':')
ax.text(25, -3.8, '−3 dB', color='#556', fontsize=8)
ax.set_ylabel('Magnitud (dB)', color='#aaa')
ax.set_ylim(-50, 5)
ax.legend(facecolor='#0d0d1a', edgecolor='#1a2030', labelcolor='#aaa', fontsize=9)
ax.set_title('Magnitud', color='#888', fontsize=10, pad=4)

# — Fase —
ax = axes[1]
ax.semilogx(f_teo,     phase_teo,          color='#f0a020', lw=2,   ls='--', label='Teórico')
ax.semilogx(f_m[mask], phase_med[mask],    color=ACCENT,    lw=1.2,          label='Medido (FRA)')
ax.axvline(FC, color='#ff6060', lw=0.8, ls=':', alpha=0.7)
ax.axhline(-45, color='#556', lw=0.6, ls=':')
ax.text(25, -47, '−45°  (en fc)', color='#556', fontsize=8)
ax.set_ylabel('Fase (°)', color='#aaa')
ax.set_ylim(-100, 10)
ax.legend(facecolor='#0d0d1a', edgecolor='#1a2030', labelcolor='#aaa', fontsize=9)
ax.set_title('Fase  (con np.unwrap para evitar saltos de ±180°)', color='#888', fontsize=10, pad=4)

# — Coherencia —
ax = axes[2]
ax.semilogx(f_m[mask], coh[mask], color='#b060e0', lw=1.2)
ax.axhline(0.99, color='#4ecb71', lw=0.8, ls='--', alpha=0.7, label='γ² = 0.99 (referencia)')
ax.fill_between(f_m[mask], coh[mask], 0, color='#b060e0', alpha=0.07)
ax.set_ylabel('Coherencia γ²', color='#aaa')
ax.set_xlabel('Frecuencia (Hz)', color='#aaa')
ax.set_ylim(0, 1.05)
ax.set_xticks([20, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 20000])
ax.set_xticklabels(['20', '50', '100', '200', '500', '1k', '2k', '5k', '10k', '20k'], color='#667')
ax.legend(facecolor='#0d0d1a', edgecolor='#1a2030', labelcolor='#aaa', fontsize=9)
ax.set_title('Coherencia — qué tan confiable es la medición en cada frecuencia  (1 = perfecta)', color='#888', fontsize=10, pad=4)

for ax in axes:
    ax.set_facecolor('#0d0d1a')
    ax.tick_params(colors='#556')
    ax.set_xlim(F_MIN, F_MAX)
    for spine in ax.spines.values(): spine.set_edgecolor('#1a2030')

plt.tight_layout()
plt.show()

## Resumen del algoritmo del FRA

```python
# Lo que corre el μC / PC en modo Bode:

x = adc1_buffer   # señal de entrada medida por ADC₁
y = adc2_buffer   # señal de salida medida por ADC₂

f, Gxy = csd(x, y, ...)   # espectro cruzado — captura la relación de fase
f, Gxx = welch(x, ...)    # autoespectro de entrada

H = Gxy / Gxx             # función de transferencia compleja

magnitud_dB = 20 * log10(|H|)               # → gráfica superior del Bode
fase_deg    = unwrap(angle(H)) * 180/π      # → gráfica inferior del Bode
coherencia  = |Gxy|² / (Gxx · Gyy)         # → confiabilidad por frecuencia
```

**Por qué `np.unwrap` en la fase:**
`angle()` devuelve valores entre −180° y +180°. Cuando la fase acumulada cruza ±180°, hay un salto artificial. `unwrap` detecta esos saltos y los corrige sumando o restando 360°, dando una curva continua.

**La coherencia es el semáforo del FRA:**
Si γ² cae por debajo de ~0.95 en alguna frecuencia, la medición en ese punto no es confiable — hay demasiado ruido, no linealidad, o el DUT no responde bien ahí.